In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

COL_PD      = '#ca3335'
COL_HEALTHY = '#467ba7'
COL_OUTLIER = '#f5a623'

In [ ]:
df = pd.read_csv('data/2025_parkinsons.csv')
df['subject'] = df['name'].str.rsplit('_', n=1).str[0]
df = df.dropna().reset_index(drop=True)

features = [c for c in df.columns if c not in ['name', 'subject', 'status']]

print(f'Recordings after dropping NaN: {len(df)}')
print(f'PD: {(df["status"]==1).sum()},  Healthy: {(df["status"]==0).sum()}')
print(f'Features: {len(features)}')

In [ ]:
# --- IQR outlier flagging ---
def iqr_flag(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)

iqr_flags = pd.DataFrame({f: iqr_flag(df[f]) for f in features}, index=df.index)
df['iqr_outlier_count'] = iqr_flags.sum(axis=1)

# --- Z-score outlier flagging ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])
z_scores = np.abs(X_scaled)
df['zscore_max']     = z_scores.max(axis=1)
df['zscore_outlier'] = (z_scores > 3).any(axis=1)

print('IQR outlier summary')
print(f"Flagged in ≥1 feature:  {(df['iqr_outlier_count'] > 0).sum()}")
print(f"Flagged in ≥3 features: {(df['iqr_outlier_count'] >= 3).sum()}")
print()
print('Z-score outlier summary (|z| > 3 in any feature)')
print(f"Total flagged: {df['zscore_outlier'].sum()}")
print()
print('Recordings flagged in ≥3 features (IQR):')
top = (df[df['iqr_outlier_count'] >= 3]
       [['name', 'status', 'iqr_outlier_count', 'zscore_max']]
       .sort_values('iqr_outlier_count', ascending=False))
print(top.to_string(index=False))

In [ ]:
# IQR outlier visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of how many features each recording is flagged in
ax = axes[0]
counts = df['iqr_outlier_count'].value_counts().sort_index()
bar_colors = ['#cccccc' if i == 0 else (COL_OUTLIER if i < 3 else COL_PD) for i in counts.index]
ax.bar(counts.index, counts.values, color=bar_colors, edgecolor='white')
ax.set_xlabel('Number of features where recording is an IQR outlier')
ax.set_ylabel('Number of recordings')
ax.set_title('Distribution of IQR Outlier Counts per Recording')

# Right: per-feature IQR outlier rate
ax2 = axes[1]
outlier_rates = iqr_flags.mean().sort_values(ascending=False) * 100
ax2.barh(outlier_rates.index, outlier_rates.values,
         color=COL_OUTLIER, edgecolor='white', height=0.7)
ax2.set_xlabel('% of recordings flagged as IQR outlier')
ax2.set_title('Per-feature IQR Outlier Rate (%)')
ax2.invert_yaxis()

fig.suptitle('IQR Outlier Analysis', fontsize=13)
plt.tight_layout()
plt.savefig('plots/outlier_iqr.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# PCA + per-group Mahalanobis distance for multivariate outlier detection
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['pc1'] = X_pca[:, 0]
df['pc2'] = X_pca[:, 1]

def mahal_dist(X):
    mean    = X.mean(axis=0)
    inv_cov = np.linalg.pinv(np.cov(X.T))
    diff    = X - mean
    return np.sqrt(np.einsum('ij,jk,ik->i', diff, inv_cov, diff))

df['mahal'] = 0.0
for g in [0, 1]:
    mask = df['status'] == g
    df.loc[mask, 'mahal'] = mahal_dist(X_pca[mask.values])

threshold = np.percentile(df['mahal'], 97.5)
df['mahal_outlier'] = df['mahal'] > threshold

# Euclidean distance from group centroid in PCA space (for comparison)
def eucl_dist(X):
    return np.sqrt(((X - X.mean(axis=0)) ** 2).sum(axis=1))

df['eucl'] = 0.0
for g in [0, 1]:
    mask = df['status'] == g
    df.loc[mask, 'eucl'] = eucl_dist(X_pca[mask.values])

eucl_threshold   = np.percentile(df['eucl'], 97.5)
df['eucl_outlier'] = df['eucl'] > eucl_threshold


In [ ]:
# Side-by-side: Mahalanobis (left) vs Euclidean (right)
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

methods = [
    ('mahal_outlier',  'mahal',  f'Mahalanobis distance\nthreshold = {threshold:.2f}  (97.5th pct)'),
    ('eucl_outlier', 'eucl',   f'Euclidean distance\nthreshold = {eucl_threshold:.2f}  (97.5th pct)'),
]

for ax, (outlier_col, dist_col, subtitle) in zip(axes, methods):
    for g, label, color in [(0, 'Healthy', COL_HEALTHY), (1, 'PD', COL_PD)]:
        m_ok  = (df['status'] == g) & ~df[outlier_col]
        m_out = (df['status'] == g) &  df[outlier_col]
        ax.scatter(df.loc[m_ok,  'pc1'], df.loc[m_ok,  'pc2'],
                   c=color, label=label, alpha=0.65, s=35, edgecolors='none')
        ax.scatter(df.loc[m_out, 'pc1'], df.loc[m_out, 'pc2'],
                   c=color, marker='X', s=110, edgecolors='black', linewidths=0.8,
                   label=f'{label} — outlier (n={int(m_out.sum())})')

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
    ax.set_title(subtitle, fontsize=11, pad=10)
    ax.legend(fontsize=8.5)

axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)

# Agreement summary between the two methods
agree_both = (df['mahal_outlier'] & df['eucl_outlier']).sum()
only_mahal = (df['mahal_outlier'] & ~df['eucl_outlier']).sum()
only_eucl  = (~df['mahal_outlier'] & df['eucl_outlier']).sum()
fig.suptitle(
    'PCA Outlier Detection — Mahalanobis vs Euclidean Distance  (97.5th percentile)\n'
    f'Agreement: both={agree_both}, Mahalanobis only={only_mahal}, Euclidean only={only_eucl}',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('plots/outlier_pca.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"Mahalanobis outliers : {df['pca_outlier'].sum()}")
print(f"Euclidean outliers: {df['eucl_outlier'].sum()}")
print(f"Flagged by both: {agree_both}")
print(f"Mahalanobis only: {only_mahal} (ellipsoidal -> accounts for feature correlation)")
print(f"Euclidean only: {only_eucl} (spherical -> treats all directions equally)")

In [ ]:
# KDE plots per feature, overlaid by group
n_cols = 4
n_rows = int(np.ceil(len(features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 2.8))
axes_flat = axes.flatten()

for i, feat in enumerate(features):
    ax = axes_flat[i]
    for g, label, color in [(0, 'Healthy', COL_HEALTHY), (1, 'PD', COL_PD)]:
        vals = df.loc[df['status'] == g, feat].dropna()
        sns.kdeplot(vals, ax=ax, color=color, fill=True,
                    alpha=0.30, linewidth=1.5, label=label)
    ax.set_title(feat, fontsize=8)
    ax.set_xlabel('')
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=6)

for j in range(len(features), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('KDE per Feature — PD (red) vs Healthy (blue)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('plots/outlier_kde_by_group.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Combined outlier flag (any of the three methods)
df['any_outlier'] = (
    (df['iqr_outlier_count'] >= 3) |
    df['zscore_outlier'] |
    df['pca_outlier']
)

print('Combined outlier summary')
print(f"  IQR (≥3 features):  {(df['iqr_outlier_count'] >= 3).sum()}")
print(f"  Z-score (|z|>3):    {df['zscore_outlier'].sum()}")
print(f"  PCA/Mahalanobis:    {df['pca_outlier'].sum()}")
print(f"  ANY of above:       {df['any_outlier'].sum()}")
print()
print('Breakdown by status:')
print(df.groupby(['status', 'any_outlier']).size().rename({0:'Healthy',1:'PD'}).unstack(fill_value=0))

- IQR flagging: identifies variable extremes in each variable (1.5 × IQR rule per feature)
- Z-score flagging (|z| > 3) catches recordings that are globally extreme after standardisation
- PCA + Mahalanobis/Euclidean distance -> captures multivariate outliers, i.e. recordings whose combination of feature values is unusual for their group even if no single feature is extreme
- KDE overlays -> full distribution shape per group